# Notebook 5 — Feature Engineering

**Task 2 — From Tables to Notebooks · Qafza Tech MLOps Training 2026/2027**

**Goal:** turn what Notebook 4 found into an actual feature table — fit every transformation on `train` only, apply it to `val` and `test`, and save every fitted object so production can reuse them exactly.

**What we'll do:**
1. Load train / val / test
2. Build the date-based features (pure arithmetic — safe to do on each split independently)
3. Decide the final feature list, explicitly excluding anything that leaks
4. Bucket rare categories (fit on train, apply everywhere)
5. Impute, scale, and one-hot encode — fit on train, apply everywhere
6. Save the feature tables, the fitted transformers, and the feature list

> ⚠️ **The rule for this whole notebook:** anything that involves *learning something from the data* (a median, a mean, a scaler's range, which categories exist) gets fit on `train_df` only, then applied — never refit — to `val_df` and `test_df`.


## 1. Load Train / Validation / Test


In [ ]:
from pathlib import Path
import json

import pandas as pd
import numpy as np

ARTIFACTS_DIR = Path("artifacts/tables")
MODELS_DIR = Path("artifacts/models")
REPORTS_DIR = Path("artifacts/reports")
MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

DATE_COLS = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

train_df = pd.read_csv(ARTIFACTS_DIR / "train.csv", parse_dates=DATE_COLS)
val_df = pd.read_csv(ARTIFACTS_DIR / "val.csv", parse_dates=DATE_COLS)
test_df = pd.read_csv(ARTIFACTS_DIR / "test.csv", parse_dates=DATE_COLS)

print("train:", train_df.shape, " val:", val_df.shape, " test:", test_df.shape)


## 2. Date Features

These are plain arithmetic on columns each row already has — no fitting involved, so it's safe to compute them identically and independently on all three splits.


In [ ]:
def add_date_features(df):
    df = df.copy()
    df["purchase_dayofweek"] = df["order_purchase_timestamp"].dt.dayofweek
    df["purchase_month"] = df["order_purchase_timestamp"].dt.month
    df["promised_delivery_days"] = (
        df["order_estimated_delivery_date"] - df["order_purchase_timestamp"]
    ).dt.days
    return df

train_df = add_date_features(train_df)
val_df = add_date_features(val_df)
test_df = add_date_features(test_df)

train_df[["order_purchase_timestamp", "order_estimated_delivery_date", "purchase_dayofweek", "purchase_month", "promised_delivery_days"]].head(3)


## 3. The Final Feature List — and What We're Leaving Out on Purpose

Only information available **at prediction time** (i.e. at purchase) is allowed in:

| Excluded column | Why |
|---|---|
| `order_status` | This is literally how the label was filtered/built |
| `order_delivered_customer_date` | The label is derived directly from this — using it would make the label trivial to "predict" |
| `order_delivered_carrier_date` | Hasn't happened yet at purchase time — it's a future event relative to prediction time |
| `order_approved_at` | Very close to purchase time in this dataset, but still technically a follow-up event — excluded to stay strict |
| `is_late` | This is the target, obviously not a feature |
| `order_id`, `customer_id` | Identifiers, not signal |

`order_estimated_delivery_date` itself isn't used directly either — we already turned it into `promised_delivery_days`, which is the useful part of it.


In [ ]:
NUMERIC_FEATURES = [
    "n_items",
    "n_distinct_products",
    "n_distinct_sellers",
    "total_price",
    "total_freight_value",
    "avg_freight_value",
    "total_weight_g",
    "max_product_length_cm",
    "max_product_height_cm",
    "max_product_width_cm",
    "total_payment_value",
    "n_payment_transactions",
    "max_payment_installments",
    "purchase_dayofweek",
    "purchase_month",
    "promised_delivery_days",
]

CATEGORICAL_FEATURES = [
    "main_product_category",
    "main_payment_type",
    "customer_state",
    "main_seller_state",
]

TARGET = "is_late"

FEATURE_COLUMNS = NUMERIC_FEATURES + CATEGORICAL_FEATURES
missing_cols = [c for c in FEATURE_COLUMNS if c not in train_df.columns]
assert not missing_cols, f"These expected feature columns are missing: {missing_cols}"
print(f"{len(NUMERIC_FEATURES)} numeric + {len(CATEGORICAL_FEATURES)} categorical = {len(FEATURE_COLUMNS)} raw feature columns")


## 4. Bucket Rare Categories in `main_product_category`

Notebook 4 showed a long tail of rare categories. We fit the "top categories" list on **train only**, then apply the same list to val/test — anything not in that list (including categories val/test might have that train never saw) becomes `"other"`.


In [ ]:
TOP_N_CATEGORIES = 20

top_categories = (
    train_df["main_product_category"].value_counts().head(TOP_N_CATEGORIES).index.tolist()
)

def bucket_rare_categories(df, column, allowed_values, other_label="other"):
    df = df.copy()
    df[column] = df[column].where(df[column].isin(allowed_values), other_label)
    df[column] = df[column].fillna(other_label)
    return df

train_df = bucket_rare_categories(train_df, "main_product_category", top_categories)
val_df = bucket_rare_categories(val_df, "main_product_category", top_categories)
test_df = bucket_rare_categories(test_df, "main_product_category", top_categories)

print("Categories kept as-is:", len(top_categories))
print(train_df["main_product_category"].value_counts().tail(5))

# This list is a fitted artifact too -- production must reuse it exactly, not recompute it.
with open(MODELS_DIR / "main_product_category_top_values.json", "w") as f:
    json.dump(top_categories, f, indent=2)


## 5. Impute, Scale, Encode — Fit on Train, Apply Everywhere

We bundle imputing + scaling (numeric) and imputing + one-hot encoding (categorical) into a single `ColumnTransformer`. Fitting it once on train and saving the *whole thing* as one artifact is simpler and safer than saving five separate pieces — there's only one object to load correctly in production.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="unknown")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, NUMERIC_FEATURES),
    ("cat", categorical_transformer, CATEGORICAL_FEATURES),
])

# Fit on TRAIN ONLY. This is the one line in the whole notebook that "learns" anything.
preprocessor.fit(train_df[FEATURE_COLUMNS])

print("Fitted. Output feature count after one-hot encoding:", len(preprocessor.get_feature_names_out()))


In [ ]:
def transform_split(df, name):
    X = preprocessor.transform(df[FEATURE_COLUMNS])
    feature_names = preprocessor.get_feature_names_out()
    X_df = pd.DataFrame(X.toarray() if hasattr(X, "toarray") else X, columns=feature_names, index=df.index)
    X_df[TARGET] = df[TARGET].values
    X_df.insert(0, "order_id", df["order_id"].values)
    print(f"{name}: {X_df.shape}")
    return X_df

train_features = transform_split(train_df, "train_features")
val_features = transform_split(val_df, "val_features")
test_features = transform_split(test_df, "test_features")

train_features.head(3)


## 6. Save Everything

The feature tables, the fitted preprocessor, and the final feature list — this is what Notebook 6 (and eventually the production pipeline) will load.


In [ ]:
import joblib

train_features.to_csv(ARTIFACTS_DIR / "train_features.csv", index=False)
val_features.to_csv(ARTIFACTS_DIR / "val_features.csv", index=False)
test_features.to_csv(ARTIFACTS_DIR / "test_features.csv", index=False)

joblib.dump(preprocessor, MODELS_DIR / "preprocessor.joblib")

feature_list = [c for c in train_features.columns if c not in ["order_id", TARGET]]
with open(REPORTS_DIR / "feature_list.json", "w") as f:
    json.dump(feature_list, f, indent=2)

print("Saved:")
print(" -", ARTIFACTS_DIR / "train_features.csv")
print(" -", ARTIFACTS_DIR / "val_features.csv")
print(" -", ARTIFACTS_DIR / "test_features.csv")
print(" -", MODELS_DIR / "preprocessor.joblib")
print(" -", MODELS_DIR / "main_product_category_top_values.json")
print(" -", REPORTS_DIR / "feature_list.json", f"({len(feature_list)} features)")


## Recap

- Built `promised_delivery_days`, `purchase_dayofweek`, `purchase_month` — pure arithmetic, computed independently on every split.
- Excluded every column that either *is* the label or only exists after the order has already shipped/arrived.
- Rare `main_product_category` values bucketed into `other`, using a list learned from `train` only.
- Missing values imputed, numeric features scaled, categoricals one-hot encoded — all fit on `train`, applied unchanged to `val`/`test`.
- **In production, this exact `preprocessor.joblib` (plus the top-categories list) gets loaded and applied — it is never refit on new data.** Refitting on new data would mean production sees a *different* set of medians/categories/scales than the model was trained on, silently breaking the model's assumptions about its own inputs.

**Next up — Notebook 6:** baseline, train, tune on validation, and touch the test set exactly once.
